# CatBoost 피처 ablation 및 제출 파일 생성

이 노트북은 저장소 준비, 라이브러리 설치, 데이터 연결, 최종 55개 피처 학습, 제출 ZIP 생성과 실행 검증까지 순서대로 진행합니다. GPU는 필요하지 않습니다.

In [ ]:
# 1. 저장소 준비
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/seokjin-wq/Aimers-9th.git'

if IN_COLAB:
    REPO_ROOT = Path('/content/Aimers-9th')
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    REPO_ROOT = next((p for p in candidates if (p / 'submission_허원준').is_dir()), current)

SUBMISSION_DIR = REPO_ROOT / 'submission_허원준'
print('저장소:', REPO_ROOT)
print('작업 폴더:', SUBMISSION_DIR)

In [ ]:
# 2. Colab 라이브러리 설치
if IN_COLAB:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SUBMISSION_DIR / 'requirements-colab.txt')],
        check=True,
    )
else:
    print('로컬 환경에서는 설치를 건너뜁니다.')

## 데이터 위치 설정

기본값은 Google Drive의 `내 드라이브/LG Aimers 9기/release data/open.zip`입니다. 다른 위치라면 `DRIVE_ZIP_PATH`만 수정합니다. 압축 파일 대신 세 CSV가 있는 폴더를 직접 지정하려면 `DATA_MODE = 'direct_dir'`로 바꾸고 `DIRECT_DATA_DIR`를 입력합니다.

In [ ]:
# 3. 데이터 위치
DATA_MODE = 'drive_zip' if IN_COLAB else 'direct_dir'  # drive_zip 또는 direct_dir
DRIVE_ZIP_PATH = '/content/drive/MyDrive/LG Aimers 9기/release data/open.zip'
DIRECT_DATA_DIR = os.environ.get('AIMERS_DATA_DIR', str(REPO_ROOT.parent / 'data'))
EXTRACT_ROOT = Path('/content/aimers_catboost_data') if IN_COLAB else SUBMISSION_DIR / '.cache' / 'data'

print('데이터 방식:', DATA_MODE)
print('Drive ZIP:', DRIVE_ZIP_PATH)
print('직접 지정 폴더:', DIRECT_DATA_DIR)

In [ ]:
# 4. 데이터 준비와 파일 확인
import shutil
import zipfile

REQUIRED_FILES = {'train.csv', 'test.csv', 'sample_submission.csv'}

def find_data_dir(root: Path) -> Path:
    root = root.expanduser().resolve()
    candidates = [root, *[p.parent for p in root.rglob('train.csv')]]
    for candidate in dict.fromkeys(candidates):
        if all((candidate / name).is_file() for name in REQUIRED_FILES):
            return candidate
    raise FileNotFoundError(f'{root} 아래에서 필요한 세 CSV를 함께 찾지 못했습니다.')

if DATA_MODE == 'drive_zip':
    if not IN_COLAB:
        raise RuntimeError('drive_zip 방식은 Colab에서 사용하세요.')
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.is_file():
        raise FileNotFoundError(f'압축 파일을 찾지 못했습니다: {zip_path}')
    if EXTRACT_ROOT.exists():
        shutil.rmtree(EXTRACT_ROOT)
    EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(EXTRACT_ROOT)
    DATA_DIR = find_data_dir(EXTRACT_ROOT)
elif DATA_MODE == 'direct_dir':
    DATA_DIR = find_data_dir(Path(DIRECT_DATA_DIR))
else:
    raise ValueError("DATA_MODE는 'drive_zip' 또는 'direct_dir'이어야 합니다.")

print('사용 데이터 폴더:', DATA_DIR)
for name in sorted(REQUIRED_FILES):
    print(f'- {name}: {(DATA_DIR / name).stat().st_size / 1024**2:.1f} MB')

## 저장된 비교 결과 확인

아래 표는 모델 변경과 피처 변경이 섞이지 않도록 같은 검증 방식으로 비교한 결과입니다. Brier는 낮을수록 좋습니다.

In [ ]:
# 5. 이미 완료된 시즌 순서 검증 결과
import pandas as pd
from IPython.display import display

brier_result = pd.read_csv(SUBMISSION_DIR / 'results' / 'brier_by_season.csv')
screen_2024 = pd.read_csv(SUBMISSION_DIR / 'results' / 'feature_screen_2024.csv')
display(brier_result)
display(screen_2024)

## 선택 사항: 피처 비교 재실행

최종 제출 파일만 만들 때는 `RUN_VALIDATION_EXPERIMENTS = False`를 유지합니다. `True`로 바꾸면 2024를 검증 시즌으로 두고 제공 47개와 최종 55개를 다시 비교합니다.

In [ ]:
# 6. 선택 실험
RUN_VALIDATION_EXPERIMENTS = False

sys.path.insert(0, str(SUBMISSION_DIR))
from src.pipeline import ModelConfig, run_experiments

if RUN_VALIDATION_EXPERIMENTS:
    validation_result = run_experiments(
        DATA_DIR,
        SUBMISSION_DIR / 'output' / 'experiments',
        experiments=['raw47_catboost', 'selected_provided41_only', 'main55_fixed'],
        validation_seasons=[2024],
        config=ModelConfig(iterations=300),
    )
    display(validation_result)
else:
    print('피처 비교 재실행을 건너뜁니다.')

## 최종 학습과 제출 ZIP 생성

2019~2024 전체 학습 데이터로 최종 55개 피처 CatBoost를 학습합니다. 생성 직후 제출 ZIP을 임시 폴더에서 실제 실행해 예측값 누락과 범위를 확인합니다.

In [ ]:
# 7. 최종 모델 생성과 제출 패키지 검증
from src.pipeline import FINAL_ITERATIONS, build_submission, validate_submission_zip

OUTPUT_DIR = SUBMISSION_DIR / 'output'
build_result = build_submission(
    DATA_DIR,
    OUTPUT_DIR,
    config=ModelConfig(iterations=FINAL_ITERATIONS),
)
validation_result = validate_submission_zip(
    DATA_DIR, OUTPUT_DIR / 'submit.zip', sample_rows=5
)

display(pd.DataFrame([build_result]))
display(pd.DataFrame([validation_result]))
print('제출 파일:', OUTPUT_DIR / 'submit.zip')

## 선택 사항: 생성 파일을 Drive에 복사

Colab 세션 종료 뒤에도 파일을 보관하려면 `COPY_TO_DRIVE = True`로 바꿉니다.

In [ ]:
# 8. 선택 저장
COPY_TO_DRIVE = False
DRIVE_OUTPUT_PATH = '/content/drive/MyDrive/LG Aimers 9기/submissions/catboost_feature_ablation_submit.zip'

if COPY_TO_DRIVE:
    destination = Path(DRIVE_OUTPUT_PATH)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(OUTPUT_DIR / 'submit.zip', destination)
    print('Drive 저장 완료:', destination)
else:
    print('Drive 복사를 건너뜁니다.')